# Feature selection: GAT-GRU

**Mục tiêu:** loại các feature không cần thiết (không mang thông tin, trùng lặp, hoặc không giúp dự báo). Lợi ích: mô hình nhỏ hơn, ít nhiễu, ít overfit và controller online cần thu ít metric hơn.

Quy trình **giống hệt** cho LSTM và GAT-GRU (code trong `modeling-common/feature_selection.py` và `forecast_torch.py`), chỉ dùng **train + val**. Tập test không được nhìn tới.

| Bước | Phương pháp | Dữ liệu |
|---|---|---|
| 1. Lọc thống kê | (a) gần hằng số: std ≈ 0 hoặc một giá trị chiếm > 99%; (b) độ liên quan = \|Spearman ρ\| giữa feature tại t và target tại t+1..t+6; (c) trùng lặp: \|ρ\| > 0,95 với feature đã giữ thì bỏ feature kém liên quan hơn | train |
| 2. Permutation importance | Huấn luyện GAT-GRU với cấu hình mặc định trên tập đã lọc. Xáo trộn từng feature (và từng nhóm) trên val, đo MAE tăng bao nhiêu. Giữ feature khi Δ > 1 độ lệch chuẩn qua các lần lặp | train → val |
| 3. Kiểm tra lại | Huấn luyện lại với tập đã chọn. Nếu val MAE xấu hơn tập đã lọc quá 2%, quay về tập đã lọc | train → val |

Feature bắt buộc: `rps_in`, `cpu_cores`, `replicas`.

**Chọn bộ dataset** bằng biến `DATASET` (đặt **giống nhau** ở mọi notebook của một lần thí nghiệm):

| `DATASET` | Nguồn | Kịch bản | Split | Thư mục đầu ra |
|---|---|---|---|---|
| `'load-testing'` | `load-testing/results/` (Locust, mô hình đóng) | normal, spike, bursty | `modeling-common/splits.json` | `processed/` |
| `'math'` | `math-load-testing/results/` (mô hình toán, luồng đến mở) | nhpp, mmpp, pareto, onoff | `modeling-common/splits_math.json` | `processed_math/` |

Hai bộ có split và thư mục kết quả riêng nên không ảnh hưởng nhau.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COMMON = (Path('..') / 'modeling-common').resolve()   # code dùng chung cho GAT-GRU và LSTM
sys.path.insert(0, str(COMMON))
import forecast_data as fd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

import feature_selection as fs

In [ ]:
DATASET = 'load-testing'          # 'load-testing' | 'math' - giống notebook tiền xử lý
TAG = '_math' if DATASET == 'math' else ''
PROCESSED = Path(f'processed{TAG}') if (Path(f'processed{TAG}') / 'runs_raw.npz').exists() else Path(f'processed{TAG}_demo')
DEMO = PROCESSED.name.endswith('_demo')
MODEL = 'gatgru'
runs, meta = fd.load_runs_npz(PROCESSED / 'runs_raw.npz')
splits = meta['splits']
edges = [tuple(e) for e in meta['edges']]
CORR_THRESHOLD = 0.95
DOMINANCE = 0.99
RETRAIN_TOLERANCE = 0.02     # bước 3: chấp nhận tập đã chọn nếu val MAE không xấu hơn quá 2%
print('PROCESSED =', PROCESSED.resolve(), '| runs =', len(runs), '| edges =', len(edges))

## Bước 1. Lọc thống kê (train)

In [ ]:
train = fd.train_arrays(runs, splits)
kept_node, report_node, corr_node = fs.filter_node_features(train, fd.HORIZON, forced=fd.FORCED_FEATURES,
                                                            dominance=DOMINANCE, corr_threshold=CORR_THRESHOLD)
print('Giữ sau bước 1:', kept_node)
report_node.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [1, 1.2]})
r = report_node.sort_values('relevance_|rho|')
colors = ['C2' if s.startswith('kept') else 'C3' for s in r.status]
axes[0].barh(r.index, r['relevance_|rho|'], color=colors)
axes[0].set_title('Độ liên quan với target tương lai (xanh = giữ, đỏ = loại)')
im = axes[1].imshow(corr_node, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr_node)), corr_node.columns, rotation=75, fontsize=8)
axes[1].set_yticks(range(len(corr_node)), corr_node.index, fontsize=8)
axes[1].set_title('Spearman ρ giữa các feature (trung bình theo service)')
fig.colorbar(im, ax=axes[1])
fig.tight_layout()
plt.show()

In [ ]:
kept_edge, report_edge, corr_edge = fs.filter_edge_features(train, edges, fd.HORIZON, dominance=DOMINANCE,
                                                            corr_threshold=CORR_THRESHOLD)
print('Feature cạnh giữ sau bước 1:', kept_edge)
report_edge.round(4)

## Bước 2. Permutation importance với GAT-GRU
Cần PyTorch (`pip install -r ../modeling-common/requirements-ml.txt`). Mô hình được huấn luyện với `DEFAULT_CONFIG` (không tune), vì mục đích chỉ là xếp hạng feature. Tune được làm sau, trên tập đã chọn.

In [ ]:
import forecast_torch as ft
B = ft.DEMO_BUDGET if DEMO else ft.BUDGET
cfg = dict(ft.DEFAULT_CONFIG[MODEL])
if DEMO:
    cfg['max_epochs'] = ft.DEMO_BUDGET['max_epochs']
data = ft.Data(runs, splits, kept_node, kept_edge)
model, val_filtered, hist, use_edges = ft.fit_one(MODEL, cfg, data, edges, seed=0)
print(f'val MAE (scaled) với tập đã lọc: {val_filtered:.4f} | epochs: {len(hist)} | params: {ft.n_params(model):,}')

groups = {}
for f in kept_node:
    groups.setdefault('group:' + fd.NODE_FEATURES[f][0], []).append(f)
if use_edges:
    groups['group:edge'] = list(kept_edge)
importance, base = ft.permutation_importance(model, data.get(cfg['window'], 'val'), data.node_feats,
                                             data.edge_feats, use_edges, groups, n_repeats=B['importance_repeats'])
importance.sort_values('delta_mean', ascending=False).round(5)

In [ ]:
imp = importance.sort_values('delta_mean')
fig, ax = plt.subplots(figsize=(9, 0.35 * len(imp) + 1.5))
ax.barh(imp.index, imp.delta_mean, xerr=imp.delta_std,
        color=['C1' if t == 'group' else 'C4' if t == 'edge' else 'C0' for t in imp.type])
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('Δ val MAE (scaled) khi xáo trộn — càng lớn càng quan trọng')
ax.set_title('Permutation importance (xanh: feature nút, tím: feature cạnh, cam: nhóm)')
plt.show()

## Bước 3. Chọn và kiểm tra lại

In [ ]:
sel_node, status_node = fs.select_by_importance(importance[importance.type == 'node'], fd.FORCED_FEATURES)
sel_edge, status_edge = (fs.select_by_importance(importance[importance.type == 'edge'], [])
                         if use_edges else (list(kept_edge), {}))
data_sel = ft.Data(runs, splits, sel_node, sel_edge)
_, val_selected, _, _ = ft.fit_one(MODEL, cfg, data_sel, edges, seed=0)
accepted = val_selected <= val_filtered * (1 + RETRAIN_TOLERANCE)
final_node, final_edge = (sel_node, sel_edge) if accepted else (kept_node, list(kept_edge))
print(f'val MAE: tập đã lọc {val_filtered:.4f} | tập đã chọn {val_selected:.4f} -> '
      f'{"dùng tập đã chọn" if accepted else "giữ tập đã lọc (tập chọn làm xấu val)"}')
print('Feature nút cuối cùng :', final_node)
print('Feature cạnh cuối cùng:', final_edge)
pd.DataFrame({'bước 1': report_node.status, 'bước 2': pd.Series(status_node)}).fillna('-')

In [ ]:
result = {
    'model': MODEL, 'node': final_node, 'edge': final_edge,
    'step1_node': report_node.reset_index(names='feature').to_dict('records'),
    'step1_edge': (report_edge.reset_index(names='feature').to_dict('records') if MODEL == 'gatgru' else []),
    'importance': importance.reset_index().to_dict('records'),
    'val_mae_filtered': val_filtered, 'val_mae_selected': val_selected, 'accepted_selection': bool(accepted),
    'config_used': cfg,
}
(PROCESSED / 'selected_features.json').write_text(json.dumps(result, indent=1, default=float))
print('Đã lưu', (PROCESSED / 'selected_features.json').resolve())